In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
import gc

# --- Configuration ---
PROCESSED_DIR = Path('../processed')
OUTPUT_DIR = Path('../processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
YEAR = 2022

# Columns needed for all calculations
REQUIRED_COLS = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'trip_distance',
    'pickup_zone', # Assuming this exists from lookup merge, else use PULocationID
    'qa_flags'
]

def process_data():
    files = sorted(PROCESSED_DIR.glob(f'yellow_tripdata_{YEAR}-*.parquet'))
    print(f"Found {len(files)} files. Starting optimized processing...")

    # --- Accumulators ---
    # High Level
    daily_stats_list = []
    
    # Granular (Viz prep)
    heatmap_parts = []
    dow_duration_parts = []
    speed_parts = []
    zone_counts_parts = []

    for f in files:
        print(f"Processing {f.name}...")
        try:
            # Load chunk
            df = pd.read_parquet(f, columns=REQUIRED_COLS)
            
            # --- Pre-calc Fields ---
            df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
            df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
            
            # Duration (minutes) & Speed (mph)
            df['duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
            df['speed_mph'] = (df['trip_distance'] / (df['duration_min'] / 60)).replace([np.inf, -np.inf], np.nan)
            
            # Time components
            df['date'] = df['tpep_pickup_datetime'].dt.date
            df['month'] = df['tpep_pickup_datetime'].dt.to_period('M')
            df['day_of_week'] = df['tpep_pickup_datetime'].dt.day_name()
            df['hour'] = df['tpep_pickup_datetime'].dt.hour

            # --- 1. Daily/Monthly Aggregation Logic ---
            # Group by date for this chunk and sum/mean
            daily_agg = df.groupby('date').agg({
                'trip_distance': 'sum',
                'duration_min': 'sum', # Will avg later
                'speed_mph': 'mean',   # Approx avg
                'tpep_pickup_datetime': 'count' # Trip count
            }).rename(columns={'tpep_pickup_datetime': 'trip_count'})
            daily_stats_list.append(daily_agg)

            # --- 2. Granular: Heatmap Data (Day + Hour) ---
            heatmap_agg = df.groupby(['day_of_week', 'hour']).size().rename('trip_count')
            heatmap_parts.append(heatmap_agg)

            # --- 3. Granular: DOW Duration (for P95 calc) ---
            # We keep raw duration columns needed for P95, minimal columns to save ram
            dow_duration_parts.append(df[['day_of_week', 'duration_min']].copy())

            # --- 4. Granular: Speed by Hour (for Median calc) ---
            speed_parts.append(df[['hour', 'speed_mph']].copy())

            # --- 5. Granular: Top Zones ---
            if 'pickup_zone' in df.columns:
                zone_agg = df['pickup_zone'].value_counts()
                zone_counts_parts.append(zone_agg)

            # Cleanup
            del df
            gc.collect()

        except Exception as e:
            print(f"Error processing {f.name}: {e}")

    # --- Final Consolidation & Saving ---
    print("Consolidating results...")

    # 1. Daily & Monthly KPIs
    all_daily = pd.concat(daily_stats_list)
    final_daily = all_daily.groupby(level=0).sum() # Re-aggregate duplicates across file boundaries
    # Recalculate averages properly if needed, but for simple aggregation:
    final_daily.to_csv(OUTPUT_DIR / f'kpi_daily_{YEAR}.csv')
    
    # Resample daily to monthly
    final_daily.index = pd.to_datetime(final_daily.index)
    final_monthly = final_daily.resample('M').sum()
    final_monthly.to_csv(OUTPUT_DIR / f'kpi_monthly_{YEAR}.csv')
    print("Saved Daily & Monthly KPIs.")

    # 2. Heatmap (Day/Hour)
    full_heatmap = pd.concat(heatmap_parts).groupby(level=[0, 1]).sum().reset_index()
    full_heatmap.to_csv(OUTPUT_DIR / 'kpi_granular_heatmap.csv', index=False)
    print("Saved Heatmap data.")

    # 3. DOW P95 Duration
    # Concatenate all raw duration data to calculate true percentile
    full_dow_dur = pd.concat(dow_duration_parts)
    p95_dow = full_dow_dur.groupby('day_of_week')['duration_min'].quantile(0.95).reset_index()
    p95_dow.columns = ['Day_of_Week', 'p95_trip_duration'] # Case sensitive for compat
    p95_dow.to_csv(OUTPUT_DIR / 'kpi_granular_dow_duration.csv', index=False)
    del full_dow_dur
    print("Saved DOW P95 data.")

    # 4. Speed by Hour (Median)
    full_speed = pd.concat(speed_parts)
    median_speed = full_speed.groupby('hour')['speed_mph'].median().reset_index()
    median_speed.columns = ['hour_of_day', 'median_speed_mph']
    median_speed.to_csv(OUTPUT_DIR / 'kpi_speed_by_hour.csv', index=False)
    del full_speed
    print("Saved Speed by Hour data.")

    # 5. Top Zones
    if zone_counts_parts:
        full_zones = pd.concat(zone_counts_parts).groupby(level=0).sum()
        full_zones.sort_values(ascending=False).head(10).to_csv(OUTPUT_DIR / 'kpi_top_zones.csv')
        print("Saved Top Zones.")

process_data()